## 09_04 - Benchmark Armstrong (dataset ridotto, tag e classi di distanza pre-istanziati)
Sostituisce 09_01/09_02/09_03/09_03a per il dataset Armstrong; **runner e configurazioni (C0-C5) restano quelli di `benchmark_runner.py`**.

**File** in `~/tesi_graphrag/data/queries/armstrong/`: `chunks_armstrong.json`, `queries_predetermined.json`, `queries_normal.json`, `eval_queries.json`, `gold_chunks.json`.
**Moduli** in `src/evaluation/`: `json_ingest.py`, `distance_presets.py`, `armstrong_curation.py`.

**Tre condizioni**, stessi tag e stesse query:
- `pre_edit`: tag attivi, nessuna distanza (sidecar `armstrong_benchmark_pre_edit.json`)
- `post_edit_expert`: distanze "da esperto di dominio" (sidecar `armstrong_benchmark.json`) = tetto del sistema
- `post_edit_user`: stessi interventi ma solo una frazione a caso (sidecar `armstrong_benchmark_user.json`) = utente medio

I tag dei chunk arrivano dal JSON; conv_tag = `expected_tag` e manual_tags = `tags` di ogni query, dal JSON.
Solo C0 (puro vettoriale) non dipende dalle distanze: si valuta una volta sola (nel `pre_edit`). Le altre config usano predq/chunk<->chunk anche senza `DIST` (il 2° retrieval parte comunque), quindi vanno rifatte nei `post_edit`.

## Ordine di esecuzione 09_04

**Prima di partire:** `COLLECTION_NAME = "armstrong_bench"` in `src/config.py`, riavvio del kernel.
JSON del dataset in `data/queries/armstrong/`; moduli `json_ingest.py`, `distance_presets.py`, `armstrong_curation.py` in `src/evaluation/`.

1. **Cella 0** (init): percorsi e parametri (`TOP_J=5`, `TOP_K=15`, `TOP_N=7`). Una volta per kernel.
2. **Cella 1** (ingestione): collezione Qdrant + tag dei chunk nel sidecar. **Una volta sola.**
3. **Cella 2** (handler + benchmark): crea `handler`, `bench`, `sm` e controlla il dataset. Da rilanciare se cambi i JSON delle query.
4. **Cella 3** (predq + similarità): registra le 18 predeterminate (**una volta per sidecar**) e lancia `check_similarity`.
   Normali con similarità >= 0.80: riformulale nel JSON, rilancia la cella 2 e il solo `check_similarity`.
5. **Cella 4**: stampa conv_tag e manual_tags per controllo.
6. *(opzionale)* **Pilota** su C0 e C1 (`PILOT_CONFIGS`). Se serve, cambia `TOP_J`/`TOP_K` **prima** del `pre_edit`.
7. **`pre_edit`**: tutte le config C0–C5, nessuna distanza. Fa il backup `armstrong_benchmark_pre_edit.json`.
   Non cambiare i parametri tra `pre_edit` e `post_edit`.
8. **Versione esperta**: genera il piano da `armstrong_curation.py`, **rivedilo**, applicalo, poi genera `post_edit_expert`.
9. **Versione utente medio**: piano ridotto da `KEEP`/`SEED`/`NOISE`, copia del sidecar `pre_edit` in `armstrong_benchmark_user.json`,
   secondo handler, poi `post_edit_user`. Per più seed: nuovo `SEED`, nuovo run_id, nuova copia del sidecar.
10. **RAGAS**: `pre_edit` con tutte le config; `post_edit_expert` e `post_edit_user` con tutte tranne C0.
    I NaN del giudice si riportano, non si sostituiscono con 0.
11. **Report**: tre condizioni per categoria, in-sample contro normali, differenze appaiate con C0 e retrieval sui gold.

**Attenzioni**
- La generazione è riprendibile; `generate()` salta ciò che è già salvato: dopo aver cambiato parametri o query usa un run_id nuovo.
- Un'unica istanza di `SidecarManager` per file (`sm = handler.sidecar_manager`).
- Solo C0 non dipende dalle distanze; le altre usano il 2° retrieval e le predq anche senza `DIST`.
- La versione esperta è un tetto: chunk, query e taratura sono scritti insieme.

In [1]:
### 0 ### Inizializzazione - una volta per kernel
import sys, json, shutil, logging
from pathlib import Path
import pandas as pd

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline_handler import PipelineHandler
from src.sidecar_manager import SidecarManager
from src.evaluation.benchmark_runner import RagasBenchmark
from src.evaluation import distance_presets as dp
from src.evaluation import armstrong_curation as cu
from src.evaluation.json_ingest import ingest_chunks_json

#logging.basicConfig(level=logging.WARNING, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", datefmt="%H:%M:%S", force=True)

COLLECTION = "armstrong_bench"
QDIR = project_root / "data/queries/armstrong"
sidecar_path = project_root / "data/processed/armstrong/armstrong_benchmark.json"
chunks_path = QDIR / "chunks_armstrong.json"
queries_path = QDIR / "eval_queries.json"
pred_path, norm_path, gold_path = QDIR / "queries_predetermined.json", QDIR / "queries_normal.json", QDIR / "gold_chunks.json"
TOP_J, TOP_K, TOP_N = 5, 15, 7   # 72 chunk: top_k=20 sarebbe un quarto del corpus

### 1 - Ingestione (UNA volta). Ricrea la collezione e scrive i tag dei chunk nel sidecar.
Prima di lanciarla: in `src/config.py` imposta `COLLECTION_NAME = "armstrong_bench"` e riavvia il kernel (l'handler legge la collezione da li').

In [ ]:
sidecar_path.parent.mkdir(parents=True, exist_ok=True)
ingest_chunks_json(chunks_path, COLLECTION, sidecar_path).release_path()   # NB: azzera i tag del sidecar e li riscrive; distanze e predq restano

In [2]:
### 2 ### Handler + benchmark (lanciare dopo l'ingestione, con COLLECTION_NAME in config.py)
handler = PipelineHandler(sidecar_path=sidecar_path, queries_path=queries_path)
sm = handler.sidecar_manager      # UNICA istanza del sidecar: usare sempre questa
bench = RagasBenchmark(handler, predetermined_path=pred_path, normal_path=norm_path,
                       out_dir=project_root / "data/benchmark/armstrong",
                       top_j=TOP_J, top_k=TOP_K, top_n=TOP_N, gold_path=gold_path)

# Controlli sul dataset
allq = json.load(open(queries_path, encoding="utf-8")); gold = json.load(open(gold_path, encoding="utf-8"))
chunks = {c["chunk_id"]: c for c in json.load(open(chunks_path, encoding="utf-8"))}
assert len({q["id"] for q in allq}) == len(allq) == 36
assert all(g in chunks for v in gold.values() for g in v), "gold con chunk inesistenti"
tags = {t for c in chunks.values() for t in c["tags"]}
assert all(q["expected_tag"] in tags | {None} and set(q["tags"]) <= tags for q in allq), "tag di query non presenti sui chunk"
print(pd.DataFrame(allq).groupby(["category", "is_predetermined"]).size().unstack())
print("tag nel sidecar:", {k: v["count"] for k, v in sm.get_global_tags().items()})

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


is_predetermined  False  True 
category                      
ambiguous             6      6
multi_domain          6      6
specialistic          6      6
tag nel sidecar: {'Jazz': 24, 'Early Life': 8, 'Career Milestones': 29, 'Technique & Skills': 11, 'Legacy & Later Life': 14, 'Adversity & Controversy': 14, 'Cycling': 24, 'Astronautics': 24}


In [ ]:
### 3 ### Registrazione delle 18 predeterminate (UNA volta per sidecar) + controllo similarita'
registered = bench.register_predetermined("pred")
print(len(registered), "predq registrate")
sim = bench.check_similarity(warn_above=0.80)
display(sim.head(8))     # le normali >= 0.80 vanno riformulate (la soglia dell'handler e' 0.85)

In [ ]:
### 4 ### Query: conv_tag e manual_tags che il runner usera' (dal JSON)
display(pd.DataFrame(allq)[["id", "category", "expected_tag", "tags", "is_predetermined"]])

### PILOTA (opzionale) - 4 query; verifica che C0 non sia al soffitto o troppo basso

In [ ]:
PILOT = ["AMB_P01", "AMB_N02", "SPE_P05", "MUL_P03"]
PILOT_CONFIGS = ["C0_baseline", "C1_tags"]            # il pilota non deve girare su tutte le config; metti None per tutte
all_cfgs = bench.configs
if PILOT_CONFIGS: bench.configs = [c for c in all_cfgs if c["name"] in PILOT_CONFIGS]
g = bench.generate("pilot", query_ids=PILOT)
bench.configs = all_cfgs
print(bench.retrieval_summary("pilot"))
for _, r in g[g.config == "C0_baseline"].iterrows():
    print(f"\n[{r.query_id}] {r.user_input}\n  RISPOSTA: {r.response[:300]}\n  REFERENCE: {r.reference[:200]}")

### 5 - `pre_edit`: tag attivi, nessuna distanza

In [ ]:
RUN_ID = "pre_edit"
gen = bench.generate(RUN_ID)
print("predq non riconosciute/errate:", gen[~gen.predq_ok].query_id.unique())
display(bench.retrieval_summary(RUN_ID))
shutil.copy(sidecar_path, sidecar_path.with_name("armstrong_benchmark_pre_edit.json"))   # backup: base per altri esperimenti

### 6 - Versione ESPERTA (tetto)
Il grafo e' tarato **come farebbe un esperto di dominio**, non a partire dal gold: `armstrong_curation.py` contiene, scritti a mano leggendo tutti i chunk,
i gruppi di chunk che raccontano la stessa storia (nucleo = classe forte, altri membri = lieve), i gruppi trasversali ai tre Armstrong,
e per ogni predq i chunk di supporto (avvicinati) e quelli fuorvianti (allontanati). Il gold serve solo a sapere qual e' la risposta di ogni predq.
In piu', con `bench`, vengono allontanati i chunk sbagliati che il retrieval vettoriale mette davvero nei top_j.
Il piano e' un JSON leggibile: **generalo, aprilo, ritoccalo (cancella/aggiungi coppie), poi applicalo**.
Attenzione: chunk, query e curation sono stati scritti insieme, quindi la curation non e' indipendente dalle query: e' un tetto, da rivedere con i tuoi occhi.
Alternativa solo-gold (ablation): `dp.build_plan(allq, gold, sm, bench=bench, cluster_union=False)`.

In [ ]:
plan_expert = dp.build_expert_plan(allq, gold, sm, cu, bench=bench)   # bench=None -> niente penalizzazioni da retrieval; include_cross=False -> solo gruppi interni
plan_expert_path = QDIR / "distance_plan_armstrong_expert.json"
dp.save_plan(plan_expert, plan_expert_path)
print(plan_expert["classes"]); print(dp.plan_stats(plan_expert)); print("-> ritocca se serve:", plan_expert_path)

In [ ]:
plan_expert = dp.load_plan(plan_expert_path)         # rilegge l'eventuale versione ritoccata a mano
print(dp.apply_plan(sm, plan_expert))                # azzera le adiacenze esistenti e scrive il piano
display(bench.edit_status().query("n_dist_edits > 0").head(20))

In [ ]:
gen = bench.generate("post_edit_expert")
print("predq non riconosciute/errate:", gen[~gen.predq_ok].query_id.unique())
display(bench.retrieval_summary("post_edit_expert"))

### 7 - Versione UTENTE MEDIO
Parte dallo stesso piano esperto e ne tiene a caso una frazione (`keep`, riproducibile con `seed`): scelte buone ma incomplete.
Va in un **altro sidecar** (copia del `pre_edit`, quindi stessi tag e stesse predq, nessuna distanza) con un secondo handler.
`class_noise` > 0 aggiunge anche scambi forte/lieve (default 0: solo omissioni).

In [ ]:
KEEP, SEED, NOISE = 0.6, 7, 0.0
plan_user = dp.degrade_plan(plan_expert, keep=KEEP, seed=SEED, class_noise=NOISE)
plan_user_path = QDIR / "distance_plan_armstrong_user.json"
dp.save_plan(plan_user, plan_user_path)
print("esperto:", dp.plan_stats(plan_expert)); print("utente :", dp.plan_stats(plan_user))

user_sidecar = sidecar_path.with_name("armstrong_benchmark_user.json")
shutil.copy(sidecar_path.with_name("armstrong_benchmark_pre_edit.json"), user_sidecar)
handler_u = PipelineHandler(sidecar_path=user_sidecar, queries_path=queries_path)
bench_u = RagasBenchmark(handler_u, predetermined_path=pred_path, normal_path=norm_path,
                         out_dir=project_root / "data/benchmark/armstrong",
                         top_j=TOP_J, top_k=TOP_K, top_n=TOP_N, gold_path=gold_path)
print(dp.apply_plan(handler_u.sidecar_manager, plan_user))
gen_u = bench_u.generate("post_edit_user")
print("predq non riconosciute/errate:", gen_u[~gen_u.predq_ok].query_id.unique())
display(bench_u.retrieval_summary("post_edit_user"))

Per ripetere con altri `keep`/`seed` (es. 5 seed per una media): cambia `KEEP, SEED`, usa un run_id nuovo (`post_edit_user_s8`) e ripeti la cella; ricordati di rifare la copia del sidecar `pre_edit`.

### 8 - RAGAS (giudice esterno). Chiavi API da variabili d'ambiente. C0 solo nel `pre_edit`; tutte le altre config in tutti e tre i run.

In [3]:
### vanno creati i seguenti moduli fittizi, non presenti in ragas per un conflitto di versioni con langchain-community
# poiché un update potrebbe danneggiare parti precedentemente funzionanti dell'architettura e vertexai non viene utilizzata
# nell'esperimento, si può tranquillamente craere un mock() per simularne l'esistenza
from unittest.mock import MagicMock

# Inganniamo Ragas facendogli credere che il modulo vertexai esista
sys.modules['langchain_community.chat_models.vertexai'] = MagicMock()
sys.modules['langchain_community.llms.vertexai'] = MagicMock()

# Ora puoi importare Ragas in sicurezza
from ragas import evaluate

In [4]:
### per far funzionare il lavoro in parallelo dei worker ragas
import nest_asyncio
nest_asyncio.apply()

In [ ]:
#from langchain_google_genai import ChatGoogleGenerativeAI
#judge_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)      # GOOGLE_API_KEY

from langchain_ollama import ChatOllama
from src.config import (
    DEFAULT_KEEP_ALIVE,
    DEFAULT_NUM_THREAD,
    LLM_MODEL,
    OLLAMA_URL,
)
judge_llm = ChatOllama(
            model=LLM_MODEL,
            base_url=OLLAMA_URL,
            keep_alive=DEFAULT_KEEP_ALIVE,
            num_thread=DEFAULT_NUM_THREAD,
            temperature=0)

NON_BASE = [c["name"] for c in bench.configs if c["mode"] != "vector"]            # tutte tranne C0
bench.evaluate_ragas("pre_edit", judge_llm, max_workers=1)                        # tutte le config
bench.evaluate_ragas("post_edit_expert", judge_llm, max_workers=1, configs=NON_BASE)
bench.evaluate_ragas("post_edit_user", judge_llm, max_workers=1, configs=NON_BASE)

Evaluating:   0%|          | 0/144 [00:00<?, ?it/s]

In [ ]:
### REPORT ###
pre = bench.load_ragas("pre_edit")
base = pre[pre.config == "C0_baseline"]      # C0 identica nei tre run
VARIANTS = {"pre_edit (nessuna distanza)": pre}
for run, label in [("post_edit_user", "utente medio"), ("post_edit_expert", "esperto")]:
    VARIANTS[f"{run} ({label})"] = pd.concat([base, bench.load_ragas(run)], ignore_index=True)

for name, df in VARIANTS.items():
    print("=====", name)
    display(bench.summary(df, by="category"))
    display(bench.summary(df, by="is_predetermined"))                    # in-sample (ritoccate) vs generalizzazione
    display(bench.paired_vs_baseline(df, subset=~df.is_predetermined))   # solo normali
print("--- retrieval su gold (nessun LLM)")
display(pd.concat({r: bench.retrieval_summary(r) for r in ["pre_edit", "post_edit_user", "post_edit_expert"]}))